# Модуль 02 · Урок 2. Логические маски и составные условия

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [1]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


pandas: 3.1.1 | корень проекта: c:\Users\304-18\Desktop\shapkina_tatiana_isip_30\pandas


## 1. Маска — отдельный объект Series
Выражение `df['price'] >= 100` не фильтрует таблицу напрямую: оно создаёт последовательность логических значений, где каждая строка получает `True` или `False`. Лишь `df.loc[mask]` оставляет строки, соответствующие `True`. Маска индексирована метками строк, поэтому при её использовании важна согласованность индексов. Для скорости и ясности сначала стройте выражение, затем применяйте его к таблице.

In [2]:
import pandas as pd
flights = pd.DataFrame({'route':['A','B','C','D'],'cost':[90,180,140,220],'delayed':[False,True,False,True]})
cheap = flights['cost'].lt(200)
print(cheap)
print(flights.loc[cheap, ['route','cost']])

0     True
1     True
2     True
3    False
Name: cost, dtype: bool
  route  cost
0     A    90
1     B   180
2     C   140


## 2. Несколько условий: & | ~ и скобки
В выражениях pandas используется `&` («И»), `|` («ИЛИ») и `~` («НЕ»), а не Python `and`, `or`, `not`. Причина: каждое сравнение возвращает целую Series, а стандартные `and`/`or` требуют одиночного логического значения. **Каждое сравнение нужно заключать в скобки** — приоритет операторов иначе может привести к ошибке или неверному ответу.

In [3]:
print(flights.loc[(flights['cost'] < 200) & (~flights['delayed'])])
print(flights.loc[(flights['route'] == 'A') | (flights['route'] == 'D')])

  route  cost  delayed
0     A    90    False
2     C   140    False
  route  cost  delayed
0     A    90    False
3     D   220     True


## 3. isin(), between() и пропуски
`isin(['A','B'])` работает как проверка в наборе значений, а `between(low,high)` включает обе границы по умолчанию. При работе с пропусками сравнения могут дать специальное значение `<NA>`, если используется nullable boolean. Для проверки пропусков предназначены `isna()` и `notna()`, а не `== None` и не `== float('nan')`.

In [4]:
members = pd.DataFrame({'tier':['silver','gold',None,'platinum'], 'score':[20,70,35,95]})
print(members.loc[members['tier'].isin(['gold','platinum'])])
print(members.loc[members['score'].between(30,80)])
print(members.loc[members['tier'].isna()])

       tier  score
1      gold     70
3  platinum     95
   tier  score
1  gold     70
2   NaN     35
  tier  score
2  NaN     35


## 4. Индекс после фильтрации
Обычная фильтрация **не перенумеровывает строки**. Это удобно для сохранения связи с источником, но может удивить при последующем `iloc` или экспорте. Если для новой таблицы нужен индекс `0, 1, 2, ...`, применяйте `reset_index(drop=True)` сознательно, не по привычке.

**Проверьте себя:** почему `and` нельзя использовать с Series? Входит ли верхняя граница в `between`? Что произойдёт с исходным индексом после фильтрации?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m2_06 · filter expensive

Выберите оплаченные заказы (`status == "paid"`) с ценой не ниже `min_price`. Сохраните все колонки и исходный индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `min_price` — входной параметр (значение видно в примере); в этом примере: `100`.


**Выполнение:** используйте готовые переменные `df`, `min_price`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [6]:
# Исходные данные задания m2_06: используются переменные df, min_price

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

min_price = 100

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_06 = expected_for("m2_06", df=df, min_price=min_price)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [7]:
# m2_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, min_price
# Сохраните конечный результат в переменную result.
result = df.loc[(df['status'] == "paid") & (df['price']>=min_price)]  # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0   paid  2026-01-03    Быстрая до...
d  demo_o4          3  Казань    книги  317.0    1         0   paid  2026-02-01          подарок
```

In [10]:
# Проверка результата предыдущей ячейки
check_result("m2_06", result, _expected_m2_06)

✓ m2_06: верно! Результат совпадает с ожидаемым выводом.


True

### m2_07 · filter by city

Оставьте строки, в которых `city` входит в `cities` и `qty > 0`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `cities` — список допустимых городов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `cities`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [11]:
# Исходные данные задания m2_07: используются переменные df, cities

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

cities = ['Москва', 'Казань']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_07 = expected_for("m2_07", df=df, cities=cities)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [18]:
# m2_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, cities
# Сохраните конечный результат в переменную result.
result = df.loc[(df['city'].isin(cities)) & (df['qty']>0)]  # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0   paid  2026-01-03    Быстрая до...
a  demo_o3          1  Москва      чай   97.0    4         5   paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0   paid  2026-02-01          подарок
```

In [19]:
# Проверка результата предыдущей ячейки
check_result("m2_07", result, _expected_m2_07)

✓ m2_07: верно! Результат совпадает с ожидаемым выводом.


True

### m2_08 · exclude status

Получите все строки за исключением статусов `bad_statuses`; сбросьте индекс с `drop=True`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `bad_statuses` — список исключаемых статусов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `bad_statuses`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [23]:
# Исходные данные задания m2_08: используются переменные df, bad_statuses

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

bad_statuses = ['cancelled', 'new']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_08 = expected_for("m2_08", df=df, bad_statuses=bad_statuses)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [26]:
# m2_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, bad_statuses
# Сохраните конечный результат в переменную result.
result = df.loc[(~df['status'].isin(bad_statuses))].reset_index(drop=True)   # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
0,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
1,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
2,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount status        date          comment
0  demo_o1          1  Москва      чай  127.0    2         0   paid  2026-01-03    Быстрая до...
1  demo_o3          1  Москва      чай   97.0    4         5   paid  2026-01-04           Срочно
2  demo_o4          3  Казань    книги  317.0    1         0   paid  2026-02-01          подарок
```

In [27]:
# Проверка результата предыдущей ячейки
check_result("m2_08", result, _expected_m2_08)

✓ m2_08: верно! Результат совпадает с ожидаемым выводом.


True

### m2_09 · price band

Отберите цены от `low` до `high` включительно, отсортируйте по `price` по убыванию.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `low` — входной параметр (значение видно в примере); в этом примере: `90`.
- `high` — входной параметр (значение видно в примере); в этом примере: `240`.


**Выполнение:** используйте готовые переменные `df`, `low`, `high`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [28]:
# Исходные данные задания m2_09: используются переменные df, low, high

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

low = 90

high = 240

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_09 = expected_for("m2_09", df=df, low=low, high=high)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [30]:
# m2_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, low, high
# Сохраните конечный результат в переменную result.
result = df.loc[df['price'].between(low,high)].sort_values(by='price', ascending=False) # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount status        date          comment
q  demo_o6          4  Казань      чай  147.0    0         0    new  2026-02-05             None
z  demo_o1          1  Москва      чай  127.0    2         0   paid  2026-01-03    Быстрая до...
a  demo_o3          1  Москва      чай   97.0    4         5   paid  2026-01-04           Срочно
```

In [31]:
# Проверка результата предыдущей ячейки
check_result("m2_09", result, _expected_m2_09)

✓ m2_09: верно! Результат совпадает с ожидаемым выводом.


True

### m2_10 · complex or

Выберите строки, где город равен `city` ИЛИ одновременно `status == "paid"` И `qty >= threshold`. Используйте скобки правильно.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `city` — входной параметр (значение видно в примере); в этом примере: `'Тверь'`.
- `threshold` — входной параметр (значение видно в примере); в этом примере: `4`.


**Выполнение:** используйте готовые переменные `df`, `city`, `threshold`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [32]:
# Исходные данные задания m2_10: используются переменные df, city, threshold

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

city = 'Тверь'

threshold = 4

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_10 = expected_for("m2_10", df=df, city=city, threshold=threshold)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [34]:
# m2_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, city, threshold
# Сохраните конечный результат в переменную result.
result = df[(df['city']==city) | ( (df['status']=="paid")&(df['qty']>= threshold))]  # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date     comment
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04  без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04      Срочно
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02            
```

In [35]:
# Проверка результата предыдущей ячейки
check_result("m2_10", result, _expected_m2_10)

✓ m2_10: верно! Результат совпадает с ожидаемым выводом.


True

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()